Download **`/sd/measurements.jsonl`** from a connected CircuitPython board over the **serial REPL** and save a copy on this machine.

Prerequisites: board connected via USB, **`pyserial`** (e.g. **`uv sync`** at repo root). The **first code cell** adds **`deploy/`** to the path the same way as [`deploy.ipynb`](deploy.ipynb) so it works from the repo root or from **`deploy/`**.

Run **Configuration** to set **`CFG.serial_port`**, or run **Pick serial** from [`deploy.ipynb`](deploy.ipynb) and reuse **`CFG`**. **Stop the CIRCUITPY drive / avoid USB mass storage** if your OS locks the port while the board is a disk — or use a workflow where the device serial stays available.

If you see **Resource busy** (errno 16), another program still has the serial port open: close a serial/REPL monitor, Thonny, `screen`/`miniterm`, `esptool`, or another terminal/Jupyter run. The download helper **retries** a few times and, on macOS, may also try the paired **`/dev/cu.…` / `/dev/tty.…` device**; if it still fails, the error may include **`lsof`** output showing which process to quit.

In [ ]:
import sys
from pathlib import Path

# So ``import notebook_env`` and ``import utils`` work from repo root or from deploy/
_repo = Path.cwd().resolve()
_deploy_candidate = _repo if (_repo / "utils.py").is_file() else _repo / "deploy"
if str(_deploy_candidate) not in sys.path:
    sys.path.insert(0, str(_deploy_candidate))

import notebook_env

notebook_env.activate()

from utils import (
    DeployConfig,
    download_circuitpy_sd_with_fallback,
    pick_serial_port_interactive,
)

CFG = DeployConfig()
OUT_DIR = Path("sd_from_device")  # folder under deploy/ if cwd is deploy/
TARGET_NAME = "measurements.jsonl"
CFG, OUT_DIR, TARGET_NAME

### Optional — pick the USB serial port (same as deploy.ipynb)
Press **Enter** to keep the current port.

In [ ]:
pick_serial_port_interactive(CFG)
CFG

### List `/sd` then download

1. **List** names in **`/sd`** (``os.listdir`` — subfolders appear too; only files should download; dirs are usually skipped on error).  
2. **Try** to save **`measurements.jsonl`** to **`OUT_DIR`**. If that file is present and it succeeds, you get that file only.  
3. If the main file is missing or fails, **download every other** name in the listing (same folder).

On-device code uses **`os.listdir` + `repr`**, and the host sends scripts with **raw REPL** (Ctrl-A, code, Ctrl-D) like **mpremote** / **Thonny** — *not* the REPL’s paste mode, which on some devices merged your script into one line (`import osD`, etc.) and never produced the end markers. Increase **`read_timeout_s`** for very large files.

In [ ]:
# Lists /sd, then tries measurements.jsonl, or the rest on failure
saved = download_circuitpy_sd_with_fallback(
    CFG.serial_port,
    OUT_DIR,
    primary=TARGET_NAME,
    root="/sd",
    read_timeout_s=300.0,
)
saved